# Agentic RAG
PDFs → chunk → embed → ChromaDB → semantic retrieval

### Imports

In [44]:
import os
import uuid
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

### 1. Load PDFs

In [45]:
def process_all_pdf(pdf_directory: str):
    """Load all PDFs in directory, tagging each page with source metadata."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    for pdf_file in pdf_dir.glob("**/*.pdf"):
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = "pdf"
        all_documents.extend(documents)

    return all_documents


all_pdf_documents = process_all_pdf("../data")
print(f"Loaded {len(all_pdf_documents)} pages")

Loaded 948 pages


### 2. Chunk

In [46]:
def split_documents(documents, chunk_size: int = 1_000, chunk_overlap: int = 200):
    """Split docs into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    return splitter.split_documents(documents)


chunks = split_documents(all_pdf_documents)
print(f"Split into {len(chunks)} chunks")

Split into 1427 chunks


### 3. Embed

In [47]:
class EmbeddingManager:
    """Sentence embeddings via SentenceTransformer."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Encode texts -> (n, dim) array."""
        return self.model.encode(texts, show_progress_bar=True)


embedding_manager = EmbeddingManager()
print(f"Embedding dim: {embedding_manager.model.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2781.60it/s]


Embedding dim: 384


C:\Users\risha\AppData\Local\Temp\ipykernel_24328\2962217753.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dim: {embedding_manager.model.get_sentence_embedding_dimension()}")


### 4. Vector Store

In [48]:
class VectorStore:
    """ChromaDB-backed vector store."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        os.makedirs(persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"description": "PDF document embeddings for RAG"},
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add docs + precomputed embeddings."""
        if len(documents) != len(embeddings):
            raise ValueError("documents and embeddings must have the same length")

        ids, metadatas, texts, emb_list = [], [], [], []
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")
            meta = dict(doc.metadata)
            meta["content_length"] = len(doc.page_content)
            metadatas.append(meta)
            texts.append(doc.page_content)
            emb_list.append(emb.tolist())

        self.collection.add(ids=ids, embeddings=emb_list, metadatas=metadatas, documents=texts)
        print(f"Added {len(documents)} documents. Total: {self.collection.count()}")


vector_store = VectorStore()
print(f"Collection already has {vector_store.collection.count()} docs")

Collection already has 1427 docs


### 5. Index

In [ ]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embeddings)

Batches:  40%|████      | 18/45 [01:11<02:42,  6.01s/it]

### 6. Retriever

In [ ]:
class RAGRetriever:
    """Semantic retrieval over a VectorStore."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Query top_k chunks above score_threshold."""
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
        )

        if not results["documents"] or not results["documents"][0]:
            return []

        retrieved = []
        for rank, (doc_id, doc, meta, dist) in enumerate(
            zip(results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]),
            start=1,
        ):
            score = 1 - dist
            if score >= score_threshold:
                retrieved.append({"id": doc_id, "content": doc, "metadata": meta, "score": score, "rank": rank})

        return retrieved


rag_retriever = RAGRetriever(vector_store, embedding_manager)

### 7. Query

In [ ]:
results = rag_retriever.retrieve("What is python?")
for r in results:
    print(f"Rank {r['rank']} | Score: {r['score']:.4f} | {r['metadata'].get('source_file')}")
    print(r["content"][:300])
    print()